## MATÍAS LANAS TAREA 5 22.466.806-6 

Parte 1

In [10]:
import numpy as np
import pandas as pd
from pathlib import Path
import pickle

In [19]:
class CatalogoEstelar:
    '''
    Clase para gestionar, limpiar y analizar datos de un catálogo estelar usando Numpy y Pandas
    '''
    def __init__(self, ruta_csv):
        #Se lee el catálogo usando pandas
        df = pd.read_csv(ruta_csv)

        #Almacenamos las columnas como arreglos independientes de numpy
        self.indicadores = df.iloc[:, 0].to_numpy(dtype=str)
        self.temperatura = df['temperatura_K'].to_numpy(dtype=float)
        self.luminosidad = df['luminosidad_Lsol'].to_numpy(dtype=float)
        self.distancia = df['distancia_pc'].to_numpy(dtype=float)

        #Ejecutamos la función de limpiar datos automáticamente
        self.limpiar_datos()

    def limpiar_datos(self):
        '''
        Limpieza de datos: elimina valores no finitos/nulos, valores no positivos y valores duplicados
        en los identificadores, manteniendo la correspondencia
        '''

        n = len(self.indicadores)

        #(a) Identificar valores finitos y estrictamente positivos en propiedades físicas
        valor_temp = np.isfinite(self.temperatura) & (self.temperatura > 0)
        valor_lum = np.isfinite(self.luminosidad) & (self.luminosidad > 0)
        valor_dist = np.isfinite(self.distancia) & (self.distancia > 0)
        mascara_numerica = valor_dist & valor_lum & valor_temp

        #(c) Manejar identificadores duplicados y conservar únicamente la primera aparición
        _, indices_unicos = np.unique(self.indicadores, return_index=True)
        mascara_no_duplicada = np.zeros(n, dtype=bool)
        mascara_no_duplicada[indices_unicos] = True

        #Mascara global combinada
        mascara_final = mascara_numerica & mascara_no_duplicada 

        #(b) Aplicar la máscara a todas las columnas
        self.indicadores = self.indicadores[mascara_final]
        self.temperatura = self.temperatura[mascara_final]
        self.luminosidad = self.luminosidad[mascara_final]
        self.distancia = self.distancia[mascara_final]

    def resumen(self):
        '''
        Análisis con Numpy: 
        '''

        #Calculamos nuestras variables necesarias
        num_estrellas = len(self.indicadores)
        temp_media = np.mean(self.temperatura)
        temp_max = np.max(self.temperatura)
        mediana_de_lum = np.median(self.luminosidad)
        dist_min = np.min(self.distancia)
        dist_max = np.min(self.distancia)

        #Obtenemos el nombre de las estrellas extremas mediante índices vectorizados
        idx_max_temp = np.argmax(self.temperatura)
        idx_min_dist = np.argmin(self.distancia)

        estrella_temp_max = self.indicadores[idx_max_temp]
        estrella_dist_min = self.indicadores[idx_min_dist]

        #Presentar los resultados claros y con sus unidades 
        print('=== Resumen estadístico del catálogo limpio ===')
        print(f'- Número total de estrellas válidas: {num_estrellas}')
        print(f'- Temperatura media: {temp_media:.2f} K')
        print(f'- Temperatura máxima: {temp_max:.2f} K ({estrella_temp_max})')
        print(f'- Luminosidad mediana: {mediana_de_lum:.2f}')
        print(f'- Distancia mínima: {dist_min:.2f} pc ({estrella_dist_min})')
        print(f'- Distancia máxima: {dist_max:.2f} pc')

    def verificar_y_crear_directorio(self):
        '''
        Método para comprobar y crear el directorio (Archivos_tarea_5)
        '''
        #Creamos el directorio usando la función Path
        carpeta = Path.cwd() / 'Archivos_tarea_5'
        carpeta.mkdir(parents=True, exist_ok=True)

    def guardar_csv(self):
        '''
        (a) Guarda el catálogo completo en formato csv dentro de Archivos_tarea_5/
        '''
        self.verificar_y_crear_directorio()

        df = pd.DataFrame({
            'ID': self.indicadores,
            'temperatura_K': self.temperatura,
            'luminosidad_Lsol': self.luminosidad,
            'distancia_pc': self.distancia
        })

        df.to_csv(Path('Archivos_tarea_5/catalogo_estelar_copia.csv'), index=False)

    def guardar_npy(self):
        '''
        (b) Guarda las temperaturas en formato .npy dentro de Archivos_tarea_5/
        '''
        self.verificar_y_crear_directorio()
        np.save(Path('Archivos_tarea_5/Temperaturas_K.npy'), self.temperatura)

    def guardar_npz(self):
        '''
        (c) Guarda todas las columnas comprimidas en formato .npz dentro de Archivos_tarea_5/
        '''
        self.verificar_y_crear_directorio()
        np.savez_compressed(
            'Archivos_tarea_5/catalogo_completo.npz',
            id=self.indicadores,
            temperatura=self.temperatura,
            luminosidad=self.luminosidad,
            distancia=self.distancia
        )

    def guardar_pickle(self):
        '''
        (d) Guarda un diccionario con todas las columnas en formato pickle
        '''
        self.verificar_y_crear_directorio()
        datos = {
            'id': self.indicadores,
            'temperatura_K': self.temperatura,
            'luminosidad_Lsol': self.luminosidad,
            'distancia_pc': self.distancia
        }

        with open(Path('Archivos_tarea_5/catalogo_completo.pkl'), 'wb') as f:
            pickle.dump(datos, f)

    def verificar_archivos(self):
        '''
        5.- (a) Verifica la integridad leyendo los archivos guardados y comparandolos con los originales
        '''

        #Primero: csv
        df_csv = pd.read_csv('Archivos_tarea_5/catalogo_estelar_copia.csv')
        ok_csv = (
            np.array_equal(self.indicadores, df_csv.iloc[:, 0].to_numpy(dtype=str)) and
            np.allclose(self.temperatura, df_csv['temperatura_K'].to_numpy(dtype=float)) and
            np.allclose(self.luminosidad, df_csv['luminosidad_Lsol'].to_numpy(dtype=float)) and
            np.allclose(self.distancia, df_csv['distancia_pc'].to_numpy(dtype=float))
        )

        print(f'- csv coincide exactamente: {ok_csv}')

        #Segundo: npy (temperaturas)
        npy_temp = np.load('Archivos_tarea_5/Temperaturas_K.npy')
        ok_npy = np.allclose(self.temperatura, npy_temp)
        print(f'- npz coincide exactamente: {ok_npy}')

        #Tercero: npz
        data_npz = np.load('Archivos_tarea_5/catalogo_completo.npz')
        ok_npz = (
            np.array_equal(self.indicadores, data_npz['id']) and
            np.allclose(self.temperatura, data_npz['temperatura']) and
            np.allclose(self.luminosidad, data_npz['luminosidad']) and
            np.allclose(self.distancia, data_npz['distancia'])
        )

        print(f'- npz coincide exactamente: {ok_npz}')

        #Cuarto: pickle
        with open(Path('Archivos_tarea_5/catalogo_completo.pkl'), 'rb') as f:
            data_pkl = pickle.load(f)

        ok_pkl = (
            np.array_equal(self.indicadores, data_pkl['id']) and
            np.allclose(self.temperatura, data_pkl['temperatura_K']) and
            np.allclose(self.luminosidad, data_pkl['luminosidad_Lsol']) and
            np.allclose(self.distancia, data_pkl['distancia_pc'])
        )

        print(f'- pkl coincide exactamente: {ok_pkl}')

    def comparar_formatos(self):
        '''
        5.- (b) Compara los tamaños en bytes de los archivos que se han generado, presentandolos en una talba muy bonita
        '''
        #Identificamos todos nuestros archivos generados con un diccionario
        archivos_generados = {
            'csv (catalogo_estelar_copia.csv)': Path('Archivos_tarea_5/catalogo_estelar_copia.csv'),
            'npy (Temperaturas_K.npy)': Path('Archivos_tarea_5/Temperaturas_K.npy'),
            'npz (catalogo_completo.npz)': Path('Archivos_tarea_5/catalogo_completo.npz'),
            'pickle (catalogo_completo.pkl)': Path('Archivos_tarea_5/catalogo_completo.pkl')
        }

        print('=== Comparación del tamaño de los archivos===')

        #Usamos una talba (tabla) de datos y un ciclo for para comparar los archivos de estos mismos
        talba_datos = []
        for nombre, ruta in archivos_generados.items():
            if ruta.exists():
                tamaño = ruta.stat().st_size
                talba_datos.append({'Formato / Archivo': nombre, 'Tamaño (bytes)': tamaño})
        df_tamaño = pd.DataFrame(talba_datos)
        print(df_tamaño.to_string(index=False))

In [20]:
#Ejecutamos nuestro precioso código
mi_catalogo = CatalogoEstelar('catalogo_estelar.csv')

#Mostramos el resumen 
mi_catalogo.resumen()

#Guardamos los datos en todos los formatos pedidos dentro de la carpeta Archivos_tarea_5
mi_catalogo.guardar_csv()
mi_catalogo.guardar_npy()
mi_catalogo.guardar_npz()
mi_catalogo.guardar_pickle()
print('\nTodos los archivos se han guardado correctamente')

#Verificamos la integridad de los archivos generados frente a los datos originales
mi_catalogo.verificar_archivos()

#Comparamos los tamaños de los archivos
mi_catalogo.comparar_formatos()

=== Resumen estadístico del catálogo limpio ===
- Número total de estrellas válidas: 92
- Temperatura media: 6388.60 K
- Temperatura máxima: 9829.36 K (E0006)
- Luminosidad mediana: 6.03
- Distancia mínima: 58.03 pc (E0039)
- Distancia máxima: 58.03 pc

Todos los archivos se han guardado correctamente
- csv coincide exactamente: True
- npz coincide exactamente: True
- npz coincide exactamente: True
- pkl coincide exactamente: True
=== Comparación del tamaño de los archivos===
               Formato / Archivo  Tamaño (bytes)
csv (catalogo_estelar_copia.csv)            5720
        npy (Temperaturas_K.npy)             864
     npz (catalogo_completo.npz)            3282
  pickle (catalogo_completo.pkl)            4385


# Preguntas teóricas

¿Cuáles son las diferencias entre CSV, NPY, NPZ y Pickle

- CSV: Es un formato de texto plano universal que llega a ser legible tanto para humanos y compatible con cualquier tipo de software, pero ocupa más espacio en el disco y necesita un mayor tiempo de procesamiento para convertir texto a números flotantes
- NPY: Es un formato binario optimizado exclusivo para arreglos de Numpy, es muy rápido de leer y escribir pero solo puede almacenar un arreglo o matriz a la vez
- NPZ: Es un contenedor comprimido optimizado por Numpy para almacenar múltiples arreglos binarios de forma eficiente y compacta
- Pickle: Es el formato estándar de ordenación de objetos de Python, ya que permite guardar estructuras de datos complejos (como diccionarios con muchos arreglos), aunque es específico del ecosistema de Python y puede llevar un riesgo al abrir archivos de fuentes no confiables

¿Qué formato utilizaría para compartir el catálogo con otros usuarios?

El formato CSV es el formato estándar universal, garantizando que cualquier usuario pueda abrirlo y leerlo tanto en entornos Python como fuera de este (Excel, LibreOffice, etc.), o distintos lenguajes de programación (C++, Java, etc.)

¿Cuál considera más apropiado para almacenar varios arreglos NumPy en un único archivo?

El formato NPZ, debido a que está diseñado especialmente para almacenar colecciones de archivos Numpy de manera eficaz, comprimida y rápida, sin depender de librerias externas o de la ordenación genérica de Python como el formato Pickle.